# Days 7-8: Fine-tuning and one comparison
**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU).

Phase 1 trained only a new head on frozen ImageNet features (validation accuracy 0.9129). Phase 2 asks: does unfreezing the later layers, with a small learning rate, beat that? It runs three experiments and records every setting and result in one table:

| name | model | what is unfrozen |
|---|---|---|
| `r18_layer4` | ResNet18, starting from your Phase 1 checkpoint | last block + head |
| `r18_layer3_4` | ResNet18, starting from your Phase 1 checkpoint | last two blocks + head |
| `mnv2_last6` | MobileNetV2 (about 3x smaller), 5 head-only epochs first, then fine-tuned | last 6 blocks + head |

For each one the table shows validation accuracy before vs after, whether the gain is statistically real (a paired test on the same 850 validation images), parameter count, file size and CPU speed, which is the accuracy-versus-size tradeoff for your write-up.

**The test set is not used.** Choosing the winner happens on validation; the single test run is Day 9.

Time: roughly 15 to 20 minutes per experiment, about 50 minutes for all three. Each finished experiment is saved to Drive; if the runtime resets, run the notebook again and it skips what's done.

## 1. Setup and data

In [ ]:
import os, time, random, zipfile, json, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision import datasets, transforms
from scipy.stats import binomtest

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU: switch the runtime to a T4 GPU")

from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/maize_ckpt"
PHASE1_CKPT = f"{CKPT_DIR}/phase1_frozen_resnet18_best.pt"
RES_PATH = f"{CKPT_DIR}/phase2_results.csv"
assert os.path.exists(PHASE1_CKPT), "Phase 1 checkpoint not found in Drive/maize_ckpt: run the Days 5-6 notebook first."

ROOT = "/content/maize_split"
if not os.path.isdir(ROOT + "/train"):
    z = "/content/drive/MyDrive/maize_split.zip"
    assert os.path.exists(z), "No /content/maize_split and no maize_split.zip on Drive: re-run Day 3 (and its optional Drive cell)."
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ROOT)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_ds = datasets.ImageFolder(ROOT + "/train", train_tf)
val_ds = datasets.ImageFolder(ROOT + "/val", eval_tf)
classes = train_ds.classes
assert classes == val_ds.classes
BATCH = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=(device == "cuda"))
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=(device == "cuda"))
print("classes:", train_ds.class_to_idx, "| train", len(train_ds), "| val", len(val_ds))


## 2. Helpers
Two details matter for fine-tuning:
- **Frozen layers stay frozen completely**, including their BatchNorm statistics: any BatchNorm layer with no trainable weights is kept in `eval()` mode.
- **Two learning rates**: a small one (`lr_bb`) for the unfrozen pretrained layers so they're nudged, not overwritten, and a larger one (`lr_head`) for the head.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def build_model(arch):
    if arch == "resnet18":
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, len(classes))
    elif arch == "mobilenet_v2":
        m = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(classes))
    else:
        raise ValueError(arch)
    return m

def head_params(m, arch):
    return list((m.fc if arch == "resnet18" else m.classifier).parameters())

def set_trainable(m, arch, unfreeze):
    for p in m.parameters():
        p.requires_grad = False
    for p in head_params(m, arch):
        p.requires_grad = True
    for path in unfreeze:                      # e.g. "layer4" or "features.13"
        mod = m
        for part in path.split("."):
            mod = mod[int(part)] if part.isdigit() else getattr(mod, part)
        for p in mod.parameters():
            p.requires_grad = True

def set_mode(m, train):
    m.train(train)
    if train:
        for mod in m.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in mod.parameters()):
                mod.eval()                     # frozen BatchNorm keeps its ImageNet statistics

def run_epoch(m, loader, train, opt=None):
    set_mode(m, train)
    n = correct = 0; loss_sum = 0.0
    preds, targets = [], []
    with torch.enable_grad() if train else torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = m(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            n += len(y); loss_sum += loss.item() * len(y)
            p = out.argmax(1); correct += (p == y).sum().item()
            preds.append(p.cpu()); targets.append(y.cpu())
    return loss_sum / n, correct / n, torch.cat(preds).numpy(), torch.cat(targets).numpy()

def run_stage(m, arch, unfreeze, lr_bb, lr_head, epochs, tag, ckpt_path):
    """Train one stage; keep the best epoch by validation accuracy (ties: lower val loss); reload it at the end."""
    set_trainable(m, arch, unfreeze)
    head = head_params(m, arch); head_ids = {id(p) for p in head}
    bb = [p for p in m.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head, "lr": lr_head}] + ([{"params": bb, "lr": lr_bb}] if bb else [])
    opt = torch.optim.Adam(groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    print(f"[{tag}] trainable parameters: {sum(p.numel() for p in head) + sum(p.numel() for p in bb):,}")
    rows, best, cm_best, preds_best = [], None, None, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta, _, _ = run_epoch(m, train_loader, True, opt)
        vl, va, vp, vt = run_epoch(m, val_loader, False)
        sched.step()
        rows.append({"stage": tag, "epoch": ep, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va, "sec": time.time() - t0})
        is_best = best is None or (va, -vl) > (best["val_acc"], -best["val_loss"])
        if is_best:
            best = rows[-1].copy()
            cm_best = np.zeros((len(classes), len(classes)), int); np.add.at(cm_best, (vt, vp), 1)
            preds_best = vp.copy()
            torch.save({"model": m.state_dict(), "arch": arch, "classes": classes, "epoch": ep, "val_acc": va}, ckpt_path)
        print(f"[{tag}] epoch {ep:2d}/{epochs}  train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time() - t0:.0f}s" + ("  <- best" if is_best else ""))
    m.load_state_dict(torch.load(ckpt_path, map_location=device)["model"])
    return best, cm_best, preds_best, pd.DataFrame(rows)

def size_and_latency(m):
    n = sum(p.numel() for p in m.parameters())
    mc = copy.deepcopy(m).cpu().eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        for _ in range(5): mc(x)
        t0 = time.time()
        for _ in range(30): mc(x)
    return n / 1e6, n * 4 / 1e6, (time.time() - t0) / 30 * 1000

def mcnemar(base_pred, new_pred, y):
    """Paired exact test on the same images: b = baseline right/new wrong, c = baseline wrong/new right."""
    b = int(((base_pred == y) & (new_pred != y)).sum()); c = int(((base_pred != y) & (new_pred == y)).sum())
    return b, c, (binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0)


## 3. The experiments
You can edit `RUN` to run only some of them (for example, drop `mnv2_last6` if short on time). The settings are the table above plus: head learning rate 3e-4, unfrozen-layer learning rate 1e-4, 10 epochs, cosine decay. Everything else matches Phase 1.

In [ ]:
EXPERIMENTS = {
    "r18_layer4":   dict(arch="resnet18", init="phase1", warmup=0, unfreeze=["layer4"], lr_bb=1e-4, lr_head=3e-4, epochs=10),
    "r18_layer3_4": dict(arch="resnet18", init="phase1", warmup=0, unfreeze=["layer3", "layer4"], lr_bb=1e-4, lr_head=3e-4, epochs=10),
    "mnv2_last6":   dict(arch="mobilenet_v2", init="imagenet", warmup=5, unfreeze=[f"features.{i}" for i in range(13, 19)], lr_bb=1e-4, lr_head=3e-4, epochs=10),
}
RUN = list(EXPERIMENTS)
print("will run:", RUN)


## 4. Run
Each experiment prints per-epoch results, then a row is appended to `phase2_results.csv` on Drive.

In [ ]:
res = pd.read_csv(RES_PATH) if os.path.exists(RES_PATH) else pd.DataFrame()
done = set(res["name"]) if len(res) else set()

for name in RUN:
    if name in done:
        print("skip (already done):", name); continue
    cfg = EXPERIMENTS[name]; arch = cfg["arch"]
    torch.manual_seed(SEED)
    m = build_model(arch).to(device)
    t_start = time.time()

    if cfg["init"] == "phase1":                   # start from the Phase 1 head
        ck = torch.load(PHASE1_CKPT, map_location=device)
        assert ck["classes"] == classes
        m.load_state_dict(ck["model"])
        _, base_acc, base_pred, y_val = run_epoch(m, val_loader, False)
        print(f"[{name}] baseline re-evaluated on val: {base_acc:.4f} (Phase 1 summary said {ck['val_acc']:.4f})")
    else:                                         # head-only warm-up gives this model its own frozen baseline
        wb, _, base_pred, _ = run_stage(m, arch, [], 0.0, 1e-3, cfg["warmup"], f"{name}:warmup", f"{CKPT_DIR}/phase2_{name}_warmup.pt")
        base_acc = wb["val_acc"]
        y_val = np.array(val_ds.targets)

    best, cm, preds, h = run_stage(m, arch, cfg["unfreeze"], cfg["lr_bb"], cfg["lr_head"], cfg["epochs"], name, f"{CKPT_DIR}/phase2_{name}_best.pt")
    b, c, p = mcnemar(base_pred, preds, y_val)
    params_m, size_mb, cpu_ms = size_and_latency(m)
    rec = cm.diagonal() / cm.sum(1)
    row = {"name": name, "arch": arch, "unfrozen": "+".join(cfg["unfreeze"]), "lr_bb": cfg["lr_bb"], "lr_head": cfg["lr_head"],
           "warmup_epochs": cfg["warmup"], "epochs": cfg["epochs"], "baseline_val_acc": base_acc,
           "best_val_acc": best["val_acc"], "best_epoch": int(best["epoch"]), "val_loss": best["val_loss"], "train_acc": best["train_acc"],
           "base_right_new_wrong": b, "base_wrong_new_right": c, "mcnemar_p": p,
           "params_M": params_m, "size_MB": size_mb, "cpu_ms_per_image": cpu_ms,
           **{f"recall_{k}": r for k, r in zip(classes, rec)}, "confusion": json.dumps(cm.tolist()),
           "minutes": (time.time() - t_start) / 60}
    res = pd.concat([res, pd.DataFrame([row])], ignore_index=True)
    res.to_csv(RES_PATH, index=False)
    h.to_csv(f"{CKPT_DIR}/phase2_{name}_history.csv", index=False)
    np.save(f"{CKPT_DIR}/phase2_{name}_val_preds.npy", preds)
    print(f"[{name}] DONE: val acc {base_acc:.4f} -> {best['val_acc']:.4f} | fixed {c}, broke {b}, p = {p:.3f} | {params_m:.1f}M params, {cpu_ms:.0f} ms/image on CPU\n")
    del m; torch.cuda.empty_cache()


## 5. Results
**How to read the test column (`p`):** it compares the new model to its own frozen baseline *on the same images*. `fixed` = images the baseline got wrong and the new model got right; `broke` = the reverse. A small `p` (under about 0.05) means the improvement is unlikely to be luck; a large one means you can't tell the two apart on this validation set, even if the raw accuracy differs.

**Accuracy vs size:** `params_M` and `size_MB` are the model's weights; `cpu_ms` is one image on Colab's CPU (rough, but good for comparing models).

In [ ]:
res = pd.read_csv(RES_PATH)
view = res[["name", "baseline_val_acc", "best_val_acc", "best_epoch", "base_right_new_wrong", "base_wrong_new_right", "mcnemar_p", "params_M", "size_MB", "cpu_ms_per_image"]].copy()
view.insert(3, "gain", (res.best_val_acc - res.baseline_val_acc).round(4))
print(view.round(4).to_string(index=False))
print("\nper-class recall of each model's best epoch:")
print(res[["name"] + [c for c in res.columns if c.startswith("recall_")]].round(3).to_string(index=False))
print("\nconfusion matrices (rows true, columns predicted; order " + ", ".join(classes) + "):")
for _, r in res.iterrows():
    print(f"  {r['name']}: {r['confusion']}")

fig, ax = plt.subplots(figsize=(7, 4))
for name in res["name"]:
    hp = f"{CKPT_DIR}/phase2_{name}_history.csv"
    if os.path.exists(hp):
        hh = pd.read_csv(hp); hh = hh[hh.stage == name]
        ax.plot(hh.epoch, hh.val_acc, marker="o", label=name)
ax.axhline(torch.load(PHASE1_CKPT, map_location="cpu")["val_acc"], color="gray", linestyle="--", label="Phase 1 frozen ResNet18")
ax.set_xlabel("fine-tuning epoch"); ax.set_ylabel("validation accuracy"); ax.legend(); ax.set_title("Phase 2 vs the frozen baseline")
plt.tight_layout(); plt.show()


## 6. Save a summary
Paste `day7_summary.md` back into the chat. Everything else (checkpoints, per-experiment histories, `phase2_results.csv`) is on your Drive in `maize_ckpt`.

In [ ]:
lines = ["# Days 7-8 summary (fine-tuning experiments, validation set only)",
         "- Phase 1 baseline: frozen ResNet18, val accuracy 0.9129 (see Day 5-6 summary)", ""]
for _, r in res.iterrows():
    lines.append(f"- {r['name']} ({r['arch']}, unfrozen: {r['unfrozen']}, lr {r['lr_bb']}/{r['lr_head']}, {int(r['epochs'])} epochs"
                 + (f", {int(r['warmup_epochs'])} head-only epochs first" if r['warmup_epochs'] else "") + "): "
                 f"val acc {r['baseline_val_acc']:.4f} -> {r['best_val_acc']:.4f} (best epoch {int(r['best_epoch'])}), "
                 f"fixed {int(r['base_wrong_new_right'])} / broke {int(r['base_right_new_wrong'])}, McNemar p = {r['mcnemar_p']:.3f}; "
                 f"{r['params_M']:.1f}M params, {r['size_MB']:.0f} MB, {r['cpu_ms_per_image']:.0f} ms/image CPU; "
                 + ", ".join(f"recall {k} {r['recall_' + k]:.3f}" for k in classes) + f"; confusion {r['confusion']}")
lines.append("")
lines.append("- notes (fill in): which experiment would you carry to Day 9, and why?")
open("/content/day7_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download("/content/day7_summary.md")
except ImportError:
    pass
